In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])
df.head()

In [ ]:
# Task 2: Write your code here:
# Do we have missing values?

cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df[cols].copy()

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

# we won't drop any feature because the data is small, however the categorical we will fill it with mode and numircal with mean
# probablily i should've used the loop to diffrentite between caregorical and numircal but i didn't want to waste time
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

for col in ['Courier_Experience_yrs', 'Delivery_Time']:
   df_clean[col] = df_clean[col].fillna(df_clean[col].mean())


 # checking
check_missing_values(df_clean)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder #import LabelEncoder
#3. Do we have categorical columns?
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

df_encoded = df_clean.copy()

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_encoded[col] = le.fit_transform(df_encoded[col])
  label_encoders[col] = le

df_encoded

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

df_scaled = df_encoded.copy()
numerical_cols = df_scaled.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET, didn't want to scale the categorical but you aksed for it :)

scaler = StandardScaler()
df_scaled[numerical_cols] = scaler.fit_transform(df_scaled[numerical_cols])
df_scaled.head()

In [ ]:
# Task 6: Write your code here:

def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df_scaled[target_column].hist()
  plt.show()

check_target_imbalance(df, "Delivery_Time")

#slightly imbalanced

In [ ]:
# Task 1: Write your code here:
X = df_scaled.drop("Delivery_Time", axis=1).astype(float)
y = df_scaled['Delivery_Time'].astype(float)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
model = RandomForestRegressor(n_estimators=200)
MAE_Results = []
# Task 2,3,4,5: Write your code here:

# Stratified K-Fold is more safe but i will use kfold becuase i had troubles trying to do stratified
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold_idx + 1}/{5}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    print(f"Training {model}... ")
    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    MAE_Results.append(mae)

    print(print(f"fold {fold_idx+1} MAE: {mae}\n"))

print(f"average MAE: {np.mean(MAE_Results)}")


In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': ["Distance_km","Weather","Traffic_Level","Time_of_Day", "Vehicle_Type", "Preparation_Time_min", "Courier_Experience_yrs"] ,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
 # Task 2: Write your code here:
y_pred.hist()

In [ ]:
# Task Bonus: Write your code here: